# 🥇 Silver → Gold
Aggregates cleaned allotment data into a cutoff table per college+branch+category+gender.

**Input:** `rankrangers_project_data.silver.mhcet_allotments`  
**Output:** `rankrangers_project_data.gold.mhcet_cutoffs`

## Cell 1 — Config

In [ ]:
CATALOG       = 'rankrangers_project_data'
SILVER_TABLE  = f'{CATALOG}.silver.mhcet_allotments'
GOLD_TABLE    = f'{CATALOG}.gold.mhcet_cutoffs'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold')
print(f'Source : {SILVER_TABLE}')
print(f'Target : {GOLD_TABLE}')

## Cell 2 — Build Gold Cutoff Table

In [ ]:
from pyspark.sql import functions as F

df_silver = spark.table(SILVER_TABLE)

# Cutoff = MIN score allotted in that round for that combo
# (lowest score that still got a seat = the floor/cutoff)
df_gold = df_silver.groupBy(
    'institute_code',
    'institute_name',
    'branch_name',
    'clean_category',
    'seat_gender'
).agg(
    # Cutoff per round
    F.round(F.min(F.when(F.col('cap_round_num') == 1, F.col('mhtcet_score'))), 4).alias('cap1_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 2, F.col('mhtcet_score'))), 4).alias('cap2_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 3, F.col('mhtcet_score'))), 4).alias('cap3_cutoff'),
    F.round(F.min(F.when(F.col('cap_round_num') == 4, F.col('mhtcet_score'))), 4).alias('cap4_cutoff'),

    # Max score per round (top of the range)
    F.round(F.max(F.when(F.col('cap_round_num') == 1, F.col('mhtcet_score'))), 4).alias('cap1_max'),
    F.round(F.max(F.when(F.col('cap_round_num') == 4, F.col('mhtcet_score'))), 4).alias('cap4_max'),

    # Total seats filled across all rounds
    F.count('*').alias('total_seats_filled'),

    # Seats per round
    F.count(F.when(F.col('cap_round_num') == 1, 1)).alias('cap1_seats'),
    F.count(F.when(F.col('cap_round_num') == 2, 1)).alias('cap2_seats'),
    F.count(F.when(F.col('cap_round_num') == 3, 1)).alias('cap3_seats'),
    F.count(F.when(F.col('cap_round_num') == 4, 1)).alias('cap4_seats'),
)

# Compute likely_round based on cap4_cutoff as primary floor
# This will be computed dynamically in the app using student's score
# but we store all cutoffs for the app to use

# Filter out combos with no cap4 cutoff (means no seats filled in final round)
df_gold = df_gold.filter(F.col('cap4_cutoff').isNotNull())

print(f'✅ Gold rows: {df_gold.count():,}')
display(df_gold.orderBy('institute_name', 'branch_name').limit(10))

## Cell 3 — Write Gold Delta Table

In [ ]:
df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', True) \
    .saveAsTable(GOLD_TABLE)

count = spark.table(GOLD_TABLE).count()
print(f'✅ Written to {GOLD_TABLE}')
print(f'   Rows: {count:,}')

## Cell 4 — Verify: Sample Query (like the app will run)

In [ ]:
# Simulate what the Streamlit app will query
# Student: OBC Male, CSE branch, score = 85.5

student_score    = 85.5
student_category = 'OBC'
student_gender   = 'M'
student_branch   = 'Computer Science and Engineering'

from pyspark.sql import functions as F

df_result = spark.table(GOLD_TABLE).filter(
    (F.col('clean_category') == student_category) &
    (F.col('seat_gender')    == student_gender) &
    (F.col('branch_name')    == student_branch) &
    (F.col('cap4_cutoff')    <= student_score)   # only show where student can get in
).withColumn(
    'likely_round',
    F.when(F.col('cap1_cutoff') <= student_score, 'CAP-I')
     .when(F.col('cap2_cutoff') <= student_score, 'CAP-II')
     .when(F.col('cap3_cutoff') <= student_score, 'CAP-III')
     .when(F.col('cap4_cutoff') <= student_score, 'CAP-IV')
     .otherwise('Unlikely')
).select(
    'institute_name', 'branch_name', 'clean_category',
    'cap1_cutoff', 'cap2_cutoff', 'cap3_cutoff', 'cap4_cutoff',
    'likely_round', 'total_seats_filled'
).orderBy('cap1_cutoff', ascending=False)

print(f'Results for: {student_branch} | {student_category} | {"Male" if student_gender=="M" else "Female"} | Score: {student_score}')
print(f'Colleges in range: {df_result.count()}')
display(df_result)

## Cell 5 — Export branch & category lists for Streamlit dropdowns

In [ ]:
import json

df_g = spark.table(GOLD_TABLE)

branches   = sorted([r[0] for r in df_g.select('branch_name').distinct().collect()])
categories = sorted([r[0] for r in df_g.select('clean_category').distinct().collect()])

print(f'Branches   ({len(branches)}):   {branches[:5]}...')
print(f'Categories ({len(categories)}): {categories}')

# Save to Volume so Streamlit app can load them
meta_path = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/app_metadata.json'
with open(meta_path, 'w') as f:
    json.dump({'branches': branches, 'categories': categories}, f)

print(f'\n✅ Metadata saved to {meta_path}')